
---

# 🌍 Applicant-Level Scholarship Probability Estimation

### Cloud-Based Objective Reasoning on Depersonalized Executive Summaries (Gemini)

---

This notebook implements the **applicant-level reasoning stage** of the AI-driven international admissions evaluation pipeline.

At this point in the workflow:

* Individual documents have already been processed locally.
* Executive summaries have been generated using a privacy-preserving local LLM.
* All personal identifying information (PII) has been removed.
* Only anonymized, policy-aligned document abstractions remain.

The present notebook aggregates those **depersonalized executive summaries** per applicant and submits them to a cloud-based reasoning agent (Google Gemini) for holistic evaluation.

---

## 🔐 Privacy Architecture

This stage operates **exclusively on depersonalized information**.

The cloud model:

* does **not** receive names, IDs, addresses, dates of birth, or any identifying attributes,
* sees only anonymized academic, institutional, and structural descriptors,
* performs reasoning purely on merit-based signals extracted in prior stages.

Thus, the system separates:

* **Identity handling (local, private)**
* **Strategic reasoning and comparative evaluation (cloud-based, anonymized)**

This architectural separation reduces privacy risk while preserving access to large-scale reasoning capabilities.

---

## 🎯 Objective of This Notebook

For each applicant, the notebook:

1. Concatenates all document-level executive summaries.
2. Sends the aggregated anonymized profile to a cloud reasoning agent.
3. Requests structured probabilistic evaluation for scholarship admission likelihood across three institutional tiers:

* **Top 10 global university**
* **Around rank 100**
* **Around rank 500**

For each tier, the agent must return:

* An integer probability (0–100%)
* 2–3 concise explanatory reasons
* Strictly valid JSON output (machine-consumable)

The output format is standardized to support:

* automated benchmarking,
* inter-applicant comparison,
* longitudinal evaluation studies,
* downstream statistical calibration.

---

## 📊 Nature of the Output

The probabilities produced by the model are:

* **heuristic estimates**, not deterministic predictions,
* reflective of typical global admissions competitiveness,
* conditioned solely on the anonymized academic and institutional signals provided.

The model is instructed to provide concise justifications to maintain interpretability and auditability.

---

## 🧠 Role in the Digital Twin Architecture

Within the broader AI admissions framework, this stage represents:

> the **strategic decision-layer reasoning agent**

It transforms structured document abstractions into **institution-tier probability signals**, which can later be fused with:

* interview transcript evaluation,
* audio-based behavioral signals,
* video-based non-verbal indicators,
* psychometric or scoring models.

The sequential execution shown here ensures reproducibility in the Colab environment.
In the full system architecture, this reasoning stage can operate in parallel with other analytical agents.

---







In [ ]:
# Mount Google Drive (Colab) and imports
import os
from pathlib import Path
import json
import time
import requests

try:
    from google.colab import drive, userdata
    if not os.path.exists('/content/drive'):
        print('Mounting Google Drive...')
        drive.mount('/content/drive')
    else:
        print('✅ Google Drive already mounted')
except Exception:
    # Not running in Colab; assume Drive is already available or user will provide path
    print('Not running in Colab or google.colab unavailable; ensure drive path is correct')
# If running in Colab, propagate the stored API key into environment variables used below
try:
    from google.colab import userdata as _userdata
    _key = _userdata.get('GOOGLE_API_KEY')
    if _key:
        #os.environ['GEMINI_API_KEY'] = _key
        os.environ['GOOGLE_API_KEY'] = _key
        print('Set GOOGLE_API_KEY from Colab userdata')
except Exception:
    pass

Mounting Google Drive...
Mounted at /content/drive
Set GOOGLE_API_KEY from Colab userdata


In [ ]:
# Configuration: adjust these as needed
# Path should match the one used by doc_executiveSummary.ipynb
DRIVE_DIR = '/content/drive/MyDrive/colab-output/dream_applicant_application/'
# Gemini model to use (changeable)
MODEL = 'gemini-flash-lite-latest'
# Output folder name template (per applicant)
OUTPUT_FOLDER_TEMPLATE = 'applicant_level_gemini_evaluations_{model}'

# Prompt template (string). Will be filled with a concatenated documents string using the placeholder {documents_text}.

url = "https://raw.githubusercontent.com/fair-vid/admission_prompts/main/evaluation/holistic_scoring.txt"
response = requests.get(url)
EVALUATION_PROMPT_TEMPLATE = response.text

# Ensure drive dir exists
Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
# Gemini client helper (streaming) - adapted from video_interviews_transcriptions_grade_info_GEMINI.ipynb
import os
try:
    from google import genai
    from google.genai import types
except Exception as e:
    print('google.genai not available in this environment. Install and configure the Google AI SDK in Colab: pip install -U google.genai')

def generate_with_gemini(filled_prompt, model=MODEL, timeout=120):
    # Prefer explicit API key from environment to avoid implicit-auth errors.
    api_key = os.environ.get('GOOGLE_API_KEY') or os.environ.get('GENAI_API_KEY') or os.environ.get('GEMINI_API_KEY')
    if not api_key:
        raise ValueError("Missing API key. Set an environment variable 'GOOGLE_API_KEY' (or 'GENAI_API_KEY'/'GEMINI_API_KEY'). In Colab you can run:\nfrom google.colab import userdata\nimport os\nos.environ['GOOGLE_API_KEY']=userdata.get('GOOGLE_API_KEY')")
    client = genai.Client(api_key=api_key)
    contents = [
        types.Content(
            role='user',
            parts=[ types.Part.from_text(text=filled_prompt) ],
        ),
    ]
    generate_content_config = types.GenerateContentConfig()
    result_text = ''
    try:
        for chunk in client.models.generate_content_stream(model=model, contents=contents, config=generate_content_config):
            if getattr(chunk, 'text', None):
                result_text += chunk.text
    except Exception as e:
        print(f'Error calling Gemini: {e}')
        return None
    return result_text

In [ ]:
# Utility: extract JSON from model response similar to doc_executiveSummary logic
import re

def extract_json_from_response(response_text):
    # Try to find JSON inside ```json blocks first
    json_pattern = r'```json\s*\n(.*?)\n```'
    match = re.search(json_pattern, response_text, re.DOTALL | re.IGNORECASE)
    if match:
        json_str = match.group(1).strip()
        try:
            parsed = json.loads(json_str)
            return True, parsed, '.json'
        except json.JSONDecodeError:
            return False, response_text, '.txt'

    # Try to parse entire response as JSON
    try:
        parsed = json.loads(response_text)
        return True, parsed, '.json'
    except json.JSONDecodeError:
        return False, response_text, '.txt'

In [ ]:
# Main aggregation and per-applicant Gemini query
from pathlib import Path

# Find applicant subdirectories (same logic as doc_executiveSummary)
drive_path = Path(DRIVE_DIR)
applicants = [p for p in drive_path.iterdir() if p.is_dir()]
print(f'Found {len(applicants)} applicant subdirectories')

for app_dir in applicants:
    # Collect all document-level summary dirs for this applicant
    doc_summary_dirs = [d for d in app_dir.iterdir() if d.is_dir() and 'document_summaries' in d.name.lower()]

    concatenated_texts = []
    for d in doc_summary_dirs:
        for f in sorted(d.iterdir()):
            if f.is_file() and f.suffix.lower() in ['.txt', '.json']:
                try:
                    text = f.read_text(encoding='utf-8')
                except Exception:
                    text = ''
                concatenated_texts.append(f'-- FILE: {f.name} --\n' + text + '\n')



    if not concatenated_texts:
        print(f'  ⊘ No document summaries found for {app_dir.name}, skipping')
        continue

    documents_text = '\n'.join(concatenated_texts)

    # Build prompt for this applicant (user-editable template)
    filled_prompt = EVALUATION_PROMPT_TEMPLATE.format(documents_text=documents_text)

    print(f'Processing applicant: {app_dir.name} (concatenated length: {len(documents_text)} chars)')

    # Query Gemini
    response_text = generate_with_gemini(filled_prompt, model=MODEL)
    if response_text is None:
        print(f'  ⚠️ Gemini returned no response for {app_dir.name}')
        continue

    # Extract JSON or save raw text
    is_json, content, ext = extract_json_from_response(response_text)

    # Prepare output directory and filename
    out_dir_name = OUTPUT_FOLDER_TEMPLATE.format(model=MODEL.replace(':', '_').replace(' ', '_'))
    out_dir = app_dir / out_dir_name
    out_dir.mkdir(parents=True, exist_ok=True)

    out_file = out_dir / f'{app_dir.name}{ext}'
    if is_json:
        # Ensure applicant_id is present (if not, add from folder name)
        if isinstance(content, dict) and 'applicant_id' not in content:
            content['applicant_id'] = app_dir.name
        with open(out_file, 'w', encoding='utf-8') as wf:
            json.dump(content, wf, ensure_ascii=False, indent=2)
    else:
        with open(out_file, 'w', encoding='utf-8') as wf:
            if isinstance(content, str):
                wf.write(content)
            else:
                wf.write(str(content))

    print(f'  ✓ Saved evaluation to {out_file}')
    # Be polite with API rate limits
    time.sleep(1)

Found 2 applicant subdirectories
  ⊘ No document summaries found for 0_0, skipping
Processing applicant: 60811_46042 (concatenated length: 3984 chars)
  ✓ Saved evaluation to /content/drive/MyDrive/colab-output/dream_applicant_application/60811_46042/applicant_level_gemini_evaluations_gemini-flash-lite-latest/60811_46042.json


Notes:
- The prompt template `EVALUATION_PROMPT_TEMPLATE` is intentionally simple; edit it to match your exact evaluation criteria.
- Ensure `GOOGLE_API_KEY` / Gemini credentials are available to `google.genai.Client()` in this environment. In Colab you can set the env var via `os.environ['GEMINI_API_KEY'] = userdata.get('GOOGLE_API_KEY')`.
- The notebook writes one file per applicant into a folder inside each applicant directory, similar to how `doc_executiveSummary.ipynb` saves outputs.